### Дообучение с небольшим числом обучаемых параметров (PEFT)

В этом ноутбуке вы будете дообучать большие языковые модели при ограниченном объёме видеопамяти. Вы реализуете prompt tuning и LoRA, а также попробуете библиотеку Hugging Face PEFT.

Для выполнения заданий нужен GPU с поддержкой CUDA. Все программные заглушки обязательной части заполнены; добавленный код отмечен комментариями `#CODEX`. Английские фразы, которые подаются модели, оставлены без перевода: они являются данными для упражнений. Названия API и переменных также сохранены.



In [ ]:
# Исходные версии библиотек
# %pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

# Версии, рекомендованные для Colab в октябре 2025 года (спасибо Льву!)
%pip install --quiet "bitsandbytes==0.45.3" "transformers>=4.43,<4.46" "accelerate>=0.33,<0.36" "peft>=0.11.1" "optimum>=1.20.0" "sentencepiece"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "Для этой части задания нужен GPU с поддержкой CUDA"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model_name = 'Enoch/llama-7b-hf'

# Загружаем токенизатор Llama...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# ...и саму модель
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # Веса — 4-битные; нормализации и активации — в fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # Сохраняем лишь часть активаций, остальные вычисляем повторно
model.enable_input_require_grads()     # Для checkpointing включаем вычисление градиентов по входам, чтобы сохранить обратное распространение
# Подробнее о gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

### Prompt tuning: история одной лисы (1 балл)

![Лиса из примера](https://i.imgur.com/Ux3qQAu.png) (источник: theodd1souts.fandom.com)


In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nРезультат:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))

Какая наглая ложь! Эта лиса уверяет, что вовсе не перепрыгивала через ленивую собаку. Она просто занималась своими делами. **Ваша задача — обучить модель говорить правду: сегодня через собаку никто не прыгал.**

В примере используется начало известной английской панграммы: «A quick brown fox» — «Быстрая бурая лиса». Целевая фраза `the_truth` отрицает привычное продолжение «jumped over the lazy dog». Для обучения используйте именно английские строки из ячеек ниже.


In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Потери:", loss)

Однако обучать всю модель мы не можем: только градиенты в формате `float32` заняли бы около 28 ГБ. Вместо этого применим [prompt tuning](https://arxiv.org/abs/2104.08691) — обучение небольшого набора векторов, добавляемых перед эмбеддингами входных токенов. Параметры исходной модели остаются замороженными.

![Схема prompt tuning](https://i.imgur.com/VwNNKnb.png)


In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    Для prompt tuning замените исходный слой эмбеддингов модели этим слоем.
    Он подставляет обучаемые векторы промпта вместо эмбеддингов первых num_prompts токенов.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # Форма input_ids: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Не забудьте добавить num_prompts PAD-токенов в начало input_ids"
        )

        #CODEX
        # Сначала получаем обычные эмбеддинги для всей последовательности.
        input_embeddings = self.original_word_embeddings(input_ids)

        #CODEX
        # expand не копирует данные, а создаёт представление промпта для каждого объекта батча.
        batch_size = input_ids.shape[0]
        learnable_prompts_expanded = self.learnable_prompts.expand(batch_size, -1, -1)

        #CODEX
        # Первые num_prompts позиций были техническими PAD-токенами — их эмбеддинги отбрасываем.
        remaining_embeddings = input_embeddings[:, self.num_prompts:, :]

        #CODEX
        # На их место ставим обучаемые векторы, сохраняя исходную длину последовательности.
        output_embeddings = torch.cat(
            [learnable_prompts_expanded, remaining_embeddings], dim=1
        )

        return output_embeddings



In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Проверки пройдены!")

**Теперь, когда слой работает,** добавим обучаемые векторы промпта в основную модель и научим её рассказывать правду о лисе.


In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "Вы уже заменили слой эмбеддингов. Если новая реализация работает неверно, заново загрузите модель"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full(
    [batch['input_ids'].shape[0], num_prompts],
    fill_value=tokenizer.pad_token_id,
    dtype=torch.int64,
    device=device,
)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

#CODEX
# Обучаем только learnable_prompts. Исходные параметры Llama были заморожены выше.
model.config.use_cache = False
model.train()
max_steps = 100
loss_threshold = 0.1

for step in range(max_steps):
    opt.zero_grad(set_to_none=True)

    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts:-1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
    loss = F.cross_entropy(
        next_word_logits.reshape(-1, next_word_logits.shape[-1]),
        true_next_tokens.reshape(-1),
    )

    loss.backward()
    opt.step()

    if step == 0 or (step + 1) % 10 == 0:
        print(f"Шаг {step + 1:3d}/{max_steps}, потери: {loss.item():.4f}")

    if loss.item() <= loss_threshold:
        print(f"Порог достигнут на шаге {step + 1}: loss={loss.item():.4f}")
        break
else:
    print("Достигнут max_steps; попробуйте увеличить число шагов.")



In [ ]:
# Итоговая проверка значения потерь
assert loss.item() <= 0.1
print("Отлично!")

In [ ]:
#CODEX
# На инференсе градиенты не нужны, а KV-cache ускоряет последовательную генерацию.
model.eval()
model.config.use_cache = True

prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

with torch.inference_mode():
    for i in range(15):
        next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
        batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
        batch['attention_mask'] = torch.cat(
            [batch['attention_mask'], torch.ones_like(next_token)], dim=-1
        )

print("\nРезультат:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().tolist()))

# Если всё сделано верно, модель будет отрицать, что лиса перепрыгнула через ленивую собаку



### Используем Hugging Face PEFT (2 балла)

[`peft`](https://huggingface.co/docs/peft/index) — библиотека для дообучения предобученных трансформеров с небольшим числом обучаемых параметров (**P**arameter **E**fficient **F**ine-**T**uning). Она предоставляет общий интерфейс для prompt tuning, prefix tuning и нескольких методов на основе адаптеров.

Перед выполнением следующей ячейки заново загрузите исходную модель из начала ноутбука: здесь нужен исходный слой эмбеддингов, без вашей обёртки `WordEmbeddingsWithLearnedPrompts`.


In [ ]:
#CODEX
# Для PEFT нужна чистая базовая модель без нашей обёртки WordEmbeddingsWithLearnedPrompts.
import gc

del model, opt, test_emb_layer, test_prompt_embeddings
gc.collect()
torch.cuda.empty_cache()

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,
)

import peft

# Стандартная подготовка к обучению 4-битной модели: замораживание базовых весов,
# корректные dtype и включение градиентов по входам для gradient checkpointing.
model = peft.prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

assert isinstance(model.model.embed_tokens, nn.Embedding), "Ожидался исходный слой эмбеддингов"

peft_config = peft.PromptTuningConfig(
    task_type=peft.TaskType.CAUSAL_LM,
    num_virtual_tokens=16,
)
model = peft.get_peft_model(model, peft_config)
model.print_trainable_parameters()



In [ ]:
# Задание: обучите модель с обёрткой PEFT так, чтобы потери при предсказании следующего токена стали меньше 0.1
# PAD-токены вручную добавлять больше не нужно, но первые num_virtual_tokens позиций logits нужно пропустить
# В конце сгенерируйте продолжение фразы и убедитесь, что модель усвоила правду

In [ ]:
# Задаём целевую фразу
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [ ]:
# Настройки обучения
loss_threshold = 0.1
num_epochs = 100

#CODEX
# Для prompt tuning обычно используют заметно больший learning rate, чем для полной модели.
learning_rate = 1e-2

#CODEX
# В оптимизатор передаём только параметры виртуального промпта.
trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=learning_rate, weight_decay=0.0)

# Задаём целевую фразу
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

#CODEX
model.config.use_cache = False
model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad(set_to_none=True)

    # PEFT самостоятельно добавляет 16 виртуальных токенов перед входной последовательностью.
    outputs = model(**batch)

    # Логиты имеют длину num_virtual_tokens + число обычных токенов.
    next_word_logits = outputs.logits[:, peft_config.num_virtual_tokens:-1, :]
    true_next_tokens = batch["input_ids"][:, 1:]

    loss = F.cross_entropy(
        next_word_logits.reshape(-1, next_word_logits.shape[-1]),
        true_next_tokens.reshape(-1),
    )

    loss.backward()
    optimizer.step()

    print(f"Эпоха {epoch + 1}/{num_epochs}, потери: {loss.item():.4f}")

    if loss.item() < loss_threshold:
        print("Целевой порог потерь достигнут. Останавливаем обучение.")
        break
else:
    print("Достигнуто максимальное число эпох; целевой порог потерь ещё не достигнут.")



In [ ]:
# Итоговая проверка: потери должны быть ниже порога
assert loss.item() < loss_threshold, "Не удалось снизить потери ниже заданного порога."
print("Обучение завершено успешно! Потери ниже 0.1.")

In [ ]:
#CODEX
model.eval()
model.config.use_cache = True

prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

with torch.inference_mode():
    for i in range(15):
        outputs = model(**batch)
        next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

        batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)
        new_attention_mask = torch.ones_like(
            next_token, dtype=batch["attention_mask"].dtype, device=device
        )
        batch["attention_mask"] = torch.cat(
            [batch["attention_mask"], new_attention_mask], dim=-1
        )

decoded_output = tokenizer.decode(
    batch["input_ids"][0].cpu().tolist(), skip_special_tokens=True
)
print("\nРезультат:", decoded_output)



### Дообучение с помощью LoRA (2 балла)

Для более содержательных задач можно использовать адаптеры низкого ранга из [статьи о LoRA](https://arxiv.org/pdf/2106.09685.pdf).

Основная идея — добавить обучаемый адаптер низкого ранга **параллельно существующему линейному слою** и сложить их выходы:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

В исходной статье LoRA адаптеры добавляли только к матрицам проекций механизма внимания. Однако [последующие работы](https://arxiv.org/abs/2305.14314) показывают, что полезно адаптировать и полносвязные блоки (FFN). Прежде чем обучать модель, реализуйте базовый слой LoRA.


In [ ]:
#CODEX
# Освобождаем PEFT-модель до создания новой копии — это важно для 16 ГБ памяти T4.
import gc

del model, optimizer, outputs, loss
gc.collect()
torch.cuda.empty_cache()

# Заново загружаем модель без ранее добавленных адаптеров PEFT.
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,
)
for param in model.parameters():
    param.requires_grad = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()



In [ ]:
class LoRALayer(nn.Module):
    """Добавляет LoRA-адаптер к существующему линейному слою."""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # Предобученный замороженный линейный слой
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        #CODEX
        # Замороженная ветвь сохраняет исходное поведение слоя.
        original_output = self.module(input)

        #CODEX
        # Низкоранговое обновление: X @ A @ B. Скобки уменьшают промежуточную размерность до rank.
        lora_output = (input @ self.adapter_A) @ self.adapter_B

        return original_output + lora_output



In [ ]:
# Проверяем вашу реализацию
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "Проверьте реализацию прямого прохода"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "Для некоторых параметров адаптера не вычислен градиент"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "Неверный градиент по матрице A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "Неверный градиент по матрице B"
# Неверный градиент: реализация отличается от формулы LoRA или нарушено вычисление градиентов (например, используется no_grad)
del dummy_loss, test_linear, test_adapter
print("Все проверки пройдены!")

### Добавляем LoRA в модель

Код ниже добавляет LoRA-адаптеры к линейным проекциям Q/K/V в механизме внимания Llama. Можно также модифицировать другие слои:

* `self_attn.o_proj` — выходную проекцию механизма внимания;
* `mlp.up_proj`, `mlp.gate_proj`, `mlp.down_proj` — полносвязные слои трансформера;
* `lm_head` — выходной слой языковой модели.

Проверка в следующей ячейке ожидает 96 адаптеров: по три проекции Q/K/V в каждом из 32 блоков Llama-7B. Дополнительные адаптеры для итогового задания добавляйте после этой проверки.


In [ ]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # Для Llama-7B

In [ ]:
#CODEX
# Переносим вход на тот же GPU, где размещена модель.
batch = tokenizer(
    "This model wants to share its greatest secret:",
    return_tensors='pt',
    return_token_type_ids=False,
).to(device)

# Проверяем один шаг обучения: градиенты должны быть ненулевыми.
out = model(**batch)
(out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Проверка градиентов пройдена, отлично!")



### Пример обучения модели

Пример ниже показывает, как обучать LoRA-адаптеры на небольшом демонстрационном датасете. В итоговом задании вам предстоит организовать похожее обучение.

**Обратите внимание:** условия итогового задания находятся ниже. Эта ячейка — только пример.


In [ ]:
# Проверяем, способна ли модель обучаться. Для полноценного обучения измените max_steps
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 примера
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # Отключаем предупреждение Trainer из Hugging Face

#CODEX
model.config.use_cache = False

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=1, # Накопление градиентов увеличивает эффективный размер батча
        warmup_steps=250, 
        max_steps=100, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1, 
        output_dir='outputs', 
        report_to=[],
        save_strategy="no" # Для совместимости по состоянию на октябрь 2025 года (спасибо Владу!) 
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# При предупреждениях о кэше задайте model.config.use_cache = False. Перед генерацией включите кэш обратно!

trainer.train()

# Это только пример: можно прервать его, не дожидаясь завершения обучения :)



### Итоговое задание: обучаем модель на практике (5 баллов)

Ваша задача — дообучить модель **генерировать код на Python**. Используйте предыдущие примеры как основу.

* **Данные:** используйте [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) или другой датасет с кодом на Python. Для этого упражнения достаточно небольшой части обучающей выборки.
* **Предобработка:** оставьте файлы с расширением `.py`. Для `codeparrot-clean` этот шаг можно пропустить: он уже содержит код на Python.
* **Длина примеров:** возьмите первые 512 символов каждого примера. Здесь ограничение задано в символах, а не в токенах.
* **Адаптеры:** используйте реализованную выше LoRA **и хотя бы одно** из следующих дополнений:
  - адаптер для `lm_head`;
  - адаптеры для компонентов MLP (`mlp.*`);
  - обучаемые входные эмбеддинги (потребуется скорректировать использование памяти).
* **Обучение:** обучать модель до сходимости не обязательно. Ориентир — 500 шагов: после этого проверьте, генерирует ли модель код методом `.generate()`. Используйте эффективный размер батча не меньше 4 и накопление градиентов `gradient_accumulation_steps=4`. Эффективный размер батча равен `per_device_train_batch_size × gradient_accumulation_steps × число GPU`.

В библиотеке `peft` тоже есть реализация LoRA. Однако для этого задания нужно показать **хотя бы один полный запуск обучения с вашей собственной реализацией LoRA**.

**Альтернативное задание:** вместо генерации кода на Python можно выбрать другой датасет и задачу, например тексты любимого исполнителя или расшифровки подкаста, если такое использование данных этично. Покажите на примерах, чему модель научилась — или не научилась, — сравнив генерации до и после дообучения, как в шаблоне ниже.


In [ ]:
#CODEX
# Промпты, на которых сравним модель до и после дообучения.
prompts = ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']

#CODEX
# Демонстрационный Trainer выше уже сделал 100 шагов на цитатах. Сбрасываем адаптеры,
# чтобы итоговый эксперимент начинался с нулевой LoRA-поправки и сравнение было честным.
for module in model.modules():
    if isinstance(module, LoRALayer):
        nn.init.kaiming_uniform_(module.adapter_A, a=5 ** 0.5)
        nn.init.zeros_(module.adapter_B)

#CODEX
# По условию добавляем как минимум один адаптер помимо Q/K/V.
# lm_head — экономичный вариант: он напрямую адаптирует распределение следующего токена.
if not isinstance(model.lm_head, LoRALayer):
    model.lm_head = LoRALayer(model.lm_head, rank=lora_rank).to(device)

trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Обучаемых параметров: {trainable_count:,} ({100 * trainable_count / total_count:.3f}%)")


#CODEX
def generate_samples(model, prompts, max_new_tokens=64, batch_size=4):
    """Детерминированно генерирует продолжения небольшими батчами, чтобы экономить память T4."""
    model.eval()
    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    generated_texts = []

    try:
        for start in range(0, len(prompts), batch_size):
            prompt_batch = prompts[start:start + batch_size]
            inputs = tokenizer(
                prompt_batch,
                return_tensors='pt',
                padding=True,
                return_token_type_ids=False,
            ).to(device)

            with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            generated_texts.extend(
                tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            )
    finally:
        tokenizer.padding_side = previous_padding_side

    return generated_texts


# Генерации до обучения — это контрольная точка для честного сравнения.
model.config.use_cache = True
generations_before = generate_samples(model, prompts)


#CODEX
# Загружаем только небольшой поток данных, а не весь датасет объёмом в десятки гигабайт.
import datasets

stream = datasets.load_dataset(
    "codeparrot/codeparrot-clean",
    split="train",
    streaming=True,
).shuffle(seed=42, buffer_size=1_000)

# 2500 примеров достаточно для 500 шагов с эффективным batch size = 4.
# Условие требует первые 512 СИМВОЛОВ, поэтому срез делаем до токенизации.
text_examples = []
for example in stream:
    code = example.get("content", "")
    if code.strip():
        text_examples.append({"text": code[:512]})
    if len(text_examples) == 2_500:
        break

train_data = datasets.Dataset.from_list(text_examples)


#CODEX
def tokenize_code(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,  # Динамический padding выполнит data collator внутри каждого батча.
        return_token_type_ids=False,
    )


train_data = train_data.map(
    tokenize_code,
    batched=True,
    remove_columns=train_data.column_names,
    desc="Токенизация Python-кода",
)

data_collator = transformers.DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


#CODEX
# 1 пример на GPU × 4 шага накопления = эффективный batch size 4, как требует условие.
model.config.use_cache = False
model.train()

training_args = transformers.TrainingArguments(
    output_dir="outputs/python_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=500,
    learning_rate=2e-4,
    warmup_steps=50,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    optim="adamw_torch",
    seed=42,
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=data_collator,
)

train_result = trainer.train()
print(train_result)


#CODEX
# После обучения возвращаем кэш для быстрой авторегрессионной генерации.
model.config.use_cache = True
generations_after = generate_samples(model, prompts)



In [ ]:
# Этот шаблон позволяет сравнить сгенерированные примеры кода в таблице.

from html import escape
from IPython.display import HTML, display

table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">ПРОМПТ</th>
    <th style="text-align: center; border:1px solid black">ДО ОБУЧЕНИЯ</th>
    <th style="text-align: center; border:1px solid black">ПОСЛЕ ОБУЧЕНИЯ</th>
  </tr>
{}
</table>"""

row_template = """  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>"""

#CODEX
# escape защищает HTML-таблицу от символов <, > и & в сгенерированном Python-коде.
rows = []
for prompt, before, after in zip(prompts, generations_before, generations_after):
    rows.append(row_template.format(escape(prompt), escape(before), escape(after)))

display(HTML(table_template.format('\n'.join(rows))))



Если вы дошли до этого места — поздравляем! Вы выполнили все основные задания.

**Дополнительное задание (бонус):** попробуйте реализовать обучение префиксов, передаваемых в механизм внимания через `model.forward(..., past_key_values=your_prompts)`. О вариантах обучения промптов и префиксов можно прочитать в статьях [1](https://arxiv.org/abs/2104.08691) и [2](https://arxiv.org/abs/2101.00190).

### Дополнительные материалы

* [Квантование после обучения](https://arxiv.org/abs/2208.07339)
* [Запуск больших моделей с помощью Accelerate](https://huggingface.co/docs/accelerate/package_reference/big_modeling)
* [Библиотека адаптеров AdapterHub](https://adapterhub.ml/)

### Запуск других моделей

Этот подход можно адаптировать к другим моделям сопоставимого размера, например [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) или [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). Потребуются небольшие изменения кода:

1. Измените имя модели в `AutoModelForCausalLM.from_pretrained()` и загрузите подходящий токенизатор через `AutoTokenizer`.
2. В коде prompt tuning замените `model.model.embed_tokens` на путь к слою эмбеддингов выбранной модели. Найти его поможет `print(model)`.
3. Адаптируйте код добавления LoRA-слоёв: названия блоков трансформера и их компонентов у других моделей отличаются.
